In [1]:
import random
import numpy as np
import torch
import pandas as pd
import os
import sys
scripts_path = os.path.abspath('../scripts')
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

from AXIS_tools import *

seed_everything(98)
#worker_init_fn(0, 0, 11)

In [2]:
# %%
from AXIS_model import *
test_fold = [0]
synergy_name = 'zip'
print(f"Test Fold {test_fold[0]} 开始")
import pandas as pd

from transformers import AutoTokenizer
import pickle
import os 
#from torch.utils.data import DataLoader, Datase

# %% [markdown]
# #### 模型定义

# %%




# %%
mlp = Predictor()

# %%
d1_t = torch.ones([4,768]) 
g_t = torch.ones([4,256])
p_t = torch.ones([4,447])
k_t = torch.ones([4,178])
m_t = torch.ones([4,225])
m_tt = torch.ones([4,1024])
geneEffect_t = torch.ones([4,256])
mlp(d1_t,d1_t,g_t,k_t,p_t,m_t,geneEffect_t,geneEffect_t,geneEffect_t,geneEffect_t,g_t,g_t)

Test Fold 0 开始


(tensor([[0.0198],
         [0.0123],
         [0.0192],
         [0.0227]], grad_fn=<DivBackward0>),
 tensor([[0.6815],
         [0.6805],
         [0.6807],
         [0.6830]], grad_fn=<SoftplusBackward0>))

In [3]:
# %%
import torch.nn as nn
import torch.nn.init as init
import math


# %%
# 初始化模型权重
#odel = param_init(model)
# 初始化模型权重
mlp = param_init(mlp)

D12C_encoder.attention_layers.0.in_proj_weight
D12C_encoder.attention_layers.0.out_proj.weight
D12C_encoder.attention_layers.1.in_proj_weight
D12C_encoder.attention_layers.1.out_proj.weight
D12C_encoder.attention_layers.2.in_proj_weight
D12C_encoder.attention_layers.2.out_proj.weight
D12C_encoder.attention_layers.3.in_proj_weight
D12C_encoder.attention_layers.3.out_proj.weight
D12C_encoder.attention_layers.4.in_proj_weight
D12C_encoder.attention_layers.4.out_proj.weight
D12C_encoder.attention_layers.5.in_proj_weight
D12C_encoder.attention_layers.5.out_proj.weight
D12C_encoder.attention_layers.6.in_proj_weight
D12C_encoder.attention_layers.6.out_proj.weight
D12C_encoder.attention_layers.7.in_proj_weight
D12C_encoder.attention_layers.7.out_proj.weight
D12C_encoder.mlp.0.weight
D12C_encoder.mlp.3.weight
CD12_encoder.attention_layers.0.in_proj_weight
CD12_encoder.attention_layers.0.out_proj.weight
CD12_encoder.attention_layers.1.in_proj_weight
CD12_encoder.attention_layers.1.out_proj.weigh

In [4]:
from torch.utils.data import DataLoader, Dataset
# %% [markdown]
# #### 模型训练

# %%
import torch
torch.cuda.is_available()

# %%
device2 = torch.device('cuda:0')
mlp.to(device2)

# %%
#参数
peft_lr = 5e-5
mlp_lr = 1e-5
batch_size = 512
test_batch = 1024
display ='430'
output_dir = f"../models/output"
log_dir = f"{output_dir}/log"

if  not os.path.exists(f"{output_dir}/result_{test_fold[0]}"):
    os.mkdir(f"{output_dir}/result_{test_fold[0]}")

# %%
#from torch.optim import AdamW
import os
from transformers import AdamW, get_linear_schedule_with_warmup,get_cosine_with_hard_restarts_schedule_with_warmup
from transformers import get_cosine_with_hard_restarts_schedule_with_warmup as warmup

#optimizer1 = AdamW(model.parameters(), lr=peft_lr,weight_decay=0.01)
optimizer2 = AdamW(mlp.parameters(), lr=mlp_lr)
from transformers import get_scheduler

from torch.utils.data import DataLoader, SequentialSampler


'''
train_dataloader = DataLoader(train_dataloader1, batch_size=batch_size,sampler=group_sampler1,drop_last=False)
eval_dataloader = DataLoader(eval_dataloader1, batch_size=test_batch, sampler=group_sampler2,drop_last=False)
test_dataloader = DataLoader(test_dataloader1, batch_size=test_batch, sampler=group_sampler3,drop_last=False)
'''


import torch
#torch.save(train_dataloader0, 'drugcombo_fold_0_data.pt')
# 加载使用 torch.save 保存的数据
# 2. 优化 Dataset 类
class mydata1(Dataset):
    def __init__(self, data):
        self.data = data
        # 预处理数据
        self._preprocess_data()
        
    def _preprocess_data(self):
        # 提前将数据转换为张量并移到CPU内存
        self.processed_data = []
        for item in self.data:
            processed_item = {
                "drug_f1": torch.as_tensor(item['drug_f1'], dtype=torch.float32),
                "drug_f2": torch.as_tensor(item['drug_f2'], dtype=torch.float32),
                "gene_f": torch.as_tensor(item['gene_f'], dtype=torch.float32),
                "protein_f": torch.as_tensor(item['protein_f'], dtype=torch.float32),
                "kegg_f": torch.as_tensor(item['kegg_f'], dtype=torch.float32),
                "meta_f": torch.as_tensor(item['meta_f'], dtype=torch.float32),
                "Bliss": torch.as_tensor([item['synergy']], dtype=torch.float32),
                "geneEffect_f": torch.as_tensor(item['geneEffect_f'], dtype=torch.float32),
                "ssGSEA_f": torch.as_tensor(item['ssGSEA_f'], dtype=torch.float32),
                "geneDependency_f": torch.as_tensor(item['geneDependency_f'], dtype=torch.float32),
                "methylation_f": torch.as_tensor(item['methylation_f'], dtype=torch.float32),
                "CNV_f": torch.as_tensor(item['CNV_f'], dtype=torch.float32),
                "mutation_f": torch.as_tensor(item['mutation_f'], dtype=torch.float32),
            }
            self.processed_data.append(processed_item)

    def __len__(self):
        return len(self.processed_data)
    
    def __getitem__(self, index):
        return self.processed_data[index]



/mnt/hpc/home/shilei/miniconda3/envs/nrf-llama2-2/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [6]:


data_fold_ls = [0,1,2,3,4,5]
tran_fold = [i for i in data_fold_ls if i not in test_fold]
data_set_dir = "../data/DrugComb"

train_0 = torch.load(f'{data_set_dir}/drugcomb_fold_{tran_fold[0]}_data.pt')
train_1 = torch.load(f'{data_set_dir}/drugcomb_fold_{tran_fold[1]}_data.pt')
train_2 = torch.load(f'{data_set_dir}/drugcomb_fold_{tran_fold[2]}_data.pt')
train_3 = torch.load(f'{data_set_dir}/drugcomb_fold_{tran_fold[3]}_data.pt')
test_0 = torch.load(f'{data_set_dir}/drugcomb_fold_{test_fold[0]}_data.pt')

from torch.utils.data import ConcatDataset

# 假设 dataset1 和 dataset2 是两个 Dataset 对象
combined_train = ConcatDataset([train_0,train_1,train_2,train_3])
#combined_loader = DataLoader(combined_dataset, batch_size=64, shuffle=True)


train_dataloader = DataLoader(
    combined_train, 
    batch_size=batch_size,
    shuffle=True)
eval_dataloader = DataLoader(
    test_0,
    batch_size=test_batch)

test_dataloader = DataLoader(
    test_0, 
    batch_size=test_batch)



len_dataset = len(combined_train)
num_epochs = 400
total_steps = (len_dataset // batch_size) * num_epochs if len_dataset % batch_size == 0 else (len_dataset // batch_size + 1) * num_epochs
warm_up_ratio = 0.005
num_training_steps = total_steps
#lr_scheduler = get_scheduler(
#    name="linear", optimizer=optimizer1, num_warmup_steps=2000, num_training_steps=num_training_steps
#)
#学习率预热
lr_scheduler2= get_scheduler(
    name="linear", optimizer=optimizer2, num_warmup_steps=2000, num_training_steps=num_training_steps
)

In [ ]:
import numpy as np

# %%
import torch
import torch.nn as nn

# %%
loss_ls = []
loss1_ls = []
loss2_ls = []
loss3_ls = []
loss4_ls = []
loss5_ls = []
tem_loss = 100
import gc
import torch
from tqdm.auto import tqdm
import shutil

ck_step = 1000

progress_bar = tqdm(range(num_training_steps))
step = 0

# 正则化参数
l1_lambda = 1e-6  # L1 正则化强度
l2_lambda = 1e-2  # L2 正则化强度


with open(f"{output_dir}/loss_new_{test_fold[0]}.txt", "w") as f:
                f.write("")
for epoch in range(num_epochs):
    for batch in train_dataloader:
        if step % ck_step == 0:
            handles1 = []
            # 遍历所有模块
        drug_f1 = batch["drug_f1"][:,0:768].to(device2)
        drug_f2 = batch["drug_f2"][:,0:768].to(device2)
        Bliss = batch[f"{synergy_name}"][:,0].to(device2)
        gene_f = batch["gene_f"][:,0:256].to(device2)
        kegg_f = batch["kegg_f"][:,0:178].to(device2)
        protein_f = batch["protein_f"][:,0:447].to(device2)
        meta_f = batch["meta_f"][:,0:225].to(device2)
        geneEffect_f = batch["geneEffect_f"].to(device2)
        ssGSEA_f = batch["ssGSEA_f"].to(device2)
        geneDependency_f = batch["geneDependency_f"].to(device2)
        methylation_f = batch["methylation_f"].to(device2)
        CNV_f = batch["CNV_f"].to(device2)
        mutation_f = batch["mutation_f"].to(device2)

        # 应用mixup
        mixed_drug_f1, mixed_drug_f2, \
        mixed_gene_f, mixed_kegg_f, mixed_protein_f, mixed_meta_f, mixed_geneEffect_f, \
        mixed_ssGSEA_f, mixed_geneDependency_f, mixed_methylation_f,mixed_CNV_f, mixed_mutation_f, y_a, y_b, lam = mixup_data(drug_f1, drug_f2,
                                gene_f, kegg_f, protein_f, meta_f, geneEffect_f, 
                                ssGSEA_f, geneDependency_f, methylation_f, CNV_f, mutation_f,
                                Bliss, alpha=1)
        
        # 使用混合数据进行前向传播
        outputs11, var11 = mlp(mixed_drug_f1, mixed_drug_f2, 
                            mixed_gene_f, mixed_kegg_f, mixed_protein_f, mixed_meta_f,mixed_geneEffect_f,mixed_ssGSEA_f, mixed_geneDependency_f, mixed_methylation_f, mixed_CNV_f, mixed_mutation_f)

        # 使用混合数据进行前向传播
        outputs22, var22 = mlp(mixed_drug_f2, mixed_drug_f1,
                            mixed_gene_f, mixed_kegg_f, mixed_protein_f, mixed_meta_f,mixed_geneEffect_f,mixed_ssGSEA_f, mixed_geneDependency_f, mixed_methylation_f, mixed_CNV_f, mixed_mutation_f)
        
        loss3 = mixup_criterion(loss_F, outputs11.view(-1), y_a, y_b, lam, var11)
        loss4 = mixup_criterion(loss_F, outputs22.view(-1), y_a, y_b, lam, var22)
                
        all_linear1_params = torch.cat([x.view(-1) for x in mlp.parameters()])
        all_linear2_params = torch.cat([x.view(-1) for x in mlp.parameters()])
        l1_regularization = l1_lambda * torch.norm(all_linear1_params, 1)
        l2_regularization = l2_lambda * torch.norm(all_linear2_params, 2)

        loss = (loss3+loss4)/2 + l1_regularization + l2_regularization 
        loss.backward()
        loss_ls.append(loss.cpu().tolist())
        # 梯度裁剪
        optimizer2.step()
        lr_scheduler2.step()
        progress_bar.update(1)
        optimizer2.zero_grad()
        step += 1
        if step % ck_step == 0:
            #测试
            torch.cuda.empty_cache()
            with torch.no_grad():
                mlp.eval()
                loss_eval = []
                for batch in test_dataloader:
                    drug_f1 = batch["drug_f1"][:,0:768].to(device2)
                    drug_f2 = batch["drug_f2"][:,0:768].to(device2)
                    Bliss = batch[f"{synergy_name}"][:,0].to(device2)
                    gene_f = batch["gene_f"][:,0:256].to(device2)
                    kegg_f = batch["kegg_f"][:,0:178].to(device2)
                    protein_f = batch["protein_f"][:,0:447].to(device2)
                    meta_f = batch["meta_f"][:,0:225].to(device2)
                    geneEffect_f = batch["geneEffect_f"].to(device2)
                    ssGSEA_f = batch["ssGSEA_f"].to(device2)
                    geneDependency_f = batch["geneDependency_f"].to(device2)
                    methylation_f = batch["methylation_f"].to(device2)
                    CNV_f = batch["CNV_f"].to(device2)
                    mutation_f = batch["mutation_f"].to(device2)
                    outputs1,var1 = mlp(drug_f1,drug_f2,gene_f,kegg_f,protein_f,meta_f,geneEffect_f,ssGSEA_f, geneDependency_f, methylation_f, CNV_f, mutation_f)
                    outputs2,var2 = mlp(drug_f2,drug_f1,gene_f,kegg_f,protein_f,meta_f,geneEffect_f,ssGSEA_f, geneDependency_f, methylation_f, CNV_f, mutation_f)
                    loss1 = torch.sqrt(loss_F2(outputs1.view(-1),Bliss))
                    loss2 = torch.sqrt(loss_F2(outputs2.view(-1),Bliss))
                    loss = (loss1+loss2)/2
                    loss_eval.append(loss.cpu().tolist())
            mean_loss_ev = np.mean(loss_eval)
            mean_loss = np.mean(loss_ls[-ck_step:])
            with open(f"{output_dir}/loss_new_{test_fold[0]}.txt", "a") as f:
                f.write(('E: {}, S: {}, TR: {:.4f}, TE: {:.4f},').format(epoch, step, mean_loss,mean_loss_ev) + "\n")
            print(('E: {}, S: {}, TR: {:.4f}, TE: {:.4f},').format(epoch, step, mean_loss,mean_loss_ev))
            torch.cuda.empty_cache()
            if (step % ck_step == 0)&(mean_loss_ev <= tem_loss):
                tem_loss = mean_loss_ev
                shutil.rmtree(f"{output_dir}/result_{test_fold[0]}/")
                if not os.path.exists(f"{output_dir}/result_{test_fold[0]}/"):
                    os.mkdir(f"{output_dir}/result_{test_fold[0]}/")
                out_path = f"{output_dir}/result_{test_fold[0]}/{step}"
                if not os.path.exists(out_path):
                    os.mkdir(out_path)
                torch.save(mlp.state_dict(),f"{out_path}/mlp_{step}.pth")
            else:
                if os.path.exists(f"{output_dir}/result_{test_fold[0]}/final"):
                    shutil.rmtree(f"{output_dir}/result_{test_fold[0]}/final")
                os.mkdir(f"{output_dir}/result_{test_fold[0]}/final")
                torch.save(mlp.state_dict(),f"{output_dir}/result_{test_fold[0]}/final/mlp_{step}.pth")
        mlp.train()
out_path = f"{output_dir}/result_{test_fold[0]}/{step}"
if not os.path.exists(out_path):
    os.mkdir(out_path)
torch.save(mlp.state_dict(),f"{out_path}/mlp_{step}.pth") 

